# Figure 1 - Panel C: 3'UTR-proportions UMAP (cell type + pseudotime)

UMAP computed from within-gene 3'UTR-site proportions (Harmony-integrated,
lineage cells). It recapitulates the cell-type structure and pseudotime
ordering, showing APA is an independent axis of cell identity, not just a
proxy for gene expression. Left: cell type; right: pseudotime. Non-lineage
cells gray.

- Backs draft **Section 1.1 (Figure 1C, Alt-2 layout)**.
- Kernel: `downstr`.
- Input: `UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_proportions_lineage.h5ad`.
- Output: `$M26_RESULTS_ROOT/Fig1/fig1c_proportions_umap/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL = "fig1c_proportions_umap"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(54)  # 85 x 54 mm (half-grid)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

PROP_UMAP_PATH = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/"
    "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_proportions_lineage.h5ad"
)

In [ ]:
adata_prop = sc.read_h5ad(PROP_UMAP_PATH)
print("proportions UMAP:", adata_prop.shape)

prop_lineage = adata_prop.obs["celltype"].isin(S.MOUSE_LINEAGE_ORDER)
prop_bg = ~prop_lineage
print(f"lineage: {int(prop_lineage.sum()):,} | non-lineage: {int(prop_bg.sum()):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
fgv = prop_lineage.values
bgv = prop_bg.values

for ax, color_key in zip(axes, ["celltype", "dpt_pseudotime"]):
    if bgv.sum() > 0:
        ax.scatter(
            adata_prop.obsm["X_umap"][bgv, 0],
            adata_prop.obsm["X_umap"][bgv, 1],
            c=S.BG_GRAY,
            s=0.5,
            alpha=0.3,
            rasterized=True,
            zorder=1,
        )
    if color_key == "celltype":
        for ct in S.MOUSE_LINEAGE_ORDER:
            m = fgv & (adata_prop.obs["celltype"] == ct).values
            if m.sum() == 0:
                continue
            ax.scatter(
                adata_prop.obsm["X_umap"][m, 0],
                adata_prop.obsm["X_umap"][m, 1],
                c=[S.paper_palette_celltypes[ct]],
                s=1.5,
                alpha=0.7,
                rasterized=True,
                zorder=2,
                label=ct,
            )
        ax.legend(
            fontsize=S.FONT_LEGEND - 0.5,
            markerscale=3,
            frameon=False,
            loc="lower left",
            handletextpad=0.1,
            borderpad=0.1,
            labelspacing=0.15,
        )
        ax.set_title("Cell type", fontsize=S.FONT_LABEL, pad=2)
    else:
        sm = ax.scatter(
            adata_prop.obsm["X_umap"][fgv, 0],
            adata_prop.obsm["X_umap"][fgv, 1],
            c=adata_prop.obs.loc[fgv, "dpt_pseudotime"],
            cmap=S.PTIME_CMAP,
            s=1.5,
            alpha=0.7,
            rasterized=True,
            zorder=2,
        )
        cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.02)
        cbar.set_label("Pseudotime", fontsize=S.FONT_TICK)
        cbar.ax.tick_params(labelsize=S.FONT_TICK - 1)
        ax.set_title("Pseudotime", fontsize=S.FONT_LABEL, pad=2)
    ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
    ax.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

axes[0].text(
    0.02,
    0.98,
    "3'UTR proportions\n(Harmony-integrated)",
    transform=axes[0].transAxes,
    fontsize=S.FONT_ANNOT - 0.5,
    va="top",
    ha="left",
    style="italic",
)
fig.subplots_adjust(wspace=0.05, left=0.02, right=0.98, top=0.92, bottom=0.10)
S.save_and_show(fig, PANEL)

In [ ]:
sc.pl.umap(
    adata_prop,
    color=["celltype", "dpt_pseudotime"],
    size=30,
    alpha=0.7,
    wspace=0.1,
    show=True,
)

In [ ]:
adata_prop.uns["neighbors"]

In [ ]:
sc.pp.neighbors(
    adata_prop, n_neighbors=15, random_state=0, n_pcs=40, use_rep="X_pca_harmony"
)
sc.tl.umap(adata_prop, min_dist=0.3, spread=1.0, random_state=42)

sc.pl.umap(
    adata_prop,
    color=["celltype", "dpt_pseudotime"],
    size=30,
    alpha=0.7,
    wspace=0.1,
    show=True,
)

In [ ]:
sc.pl.umap(
    adata_prop,
    color=["celltype", "dpt_pseudotime", "batch_number", "phase"],
    size=30,
    alpha=0.7,
    wspace=0.1,
    ncols=2,
    show=True,
)

In [ ]:
PANEL = "fig1c_proportions_umap_v2"

print("proportions UMAP:", adata_prop.shape)

prop_lineage = adata_prop.obs["celltype"].isin(S.MOUSE_LINEAGE_ORDER)
prop_bg = ~prop_lineage
print(f"lineage: {int(prop_lineage.sum()):,} | non-lineage: {int(prop_bg.sum()):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
fgv = prop_lineage.values
bgv = prop_bg.values

for ax, color_key in zip(axes, ["celltype", "dpt_pseudotime"]):
    if bgv.sum() > 0:
        ax.scatter(
            adata_prop.obsm["X_umap"][bgv, 0],
            adata_prop.obsm["X_umap"][bgv, 1],
            c=S.BG_GRAY,
            s=0.5,
            alpha=0.3,
            rasterized=True,
            zorder=1,
        )
    if color_key == "celltype":
        for ct in S.MOUSE_LINEAGE_ORDER:
            m = fgv & (adata_prop.obs["celltype"] == ct).values
            if m.sum() == 0:
                continue
            ax.scatter(
                adata_prop.obsm["X_umap"][m, 0],
                adata_prop.obsm["X_umap"][m, 1],
                c=[S.paper_palette_celltypes[ct]],
                s=1.5,
                alpha=0.7,
                rasterized=True,
                zorder=2,
                label=ct,
            )
        ax.legend(
            fontsize=S.FONT_LEGEND - 0.5,
            markerscale=3,
            frameon=False,
            loc="lower right",
            handletextpad=0.1,
            borderpad=0.1,
            labelspacing=0.15,
        )
        ax.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
        ax.set_title("Cell type", fontsize=S.FONT_LABEL, pad=2)
    else:
        sm = ax.scatter(
            adata_prop.obsm["X_umap"][fgv, 0],
            adata_prop.obsm["X_umap"][fgv, 1],
            c=adata_prop.obs.loc[fgv, "dpt_pseudotime"],
            cmap=S.PTIME_CMAP,
            s=1.5,
            alpha=0.7,
            rasterized=True,
            zorder=2,
        )
        cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.02)
        cbar.set_label("Pseudotime", fontsize=S.FONT_TICK)
        cbar.ax.tick_params(labelsize=S.FONT_TICK - 1)
        ax.set_title("Pseudotime", fontsize=S.FONT_LABEL, pad=2)
    ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

axes[0].text(
    0.50,
    0.98,
    "3'UTR proportions\n(Harmony-integrated)",
    transform=axes[0].transAxes,
    fontsize=S.FONT_ANNOT - 0.5,
    va="top",
    ha="left",
    style="italic",
)
fig.subplots_adjust(wspace=0.05, left=0.05, right=0.9, top=0.92, bottom=0.10)
S.save_and_show(fig, PANEL)

### Chosen variant, random-order + reduced points (TODO 4)

The cell below re-plots the chosen `_v2` embedding with cells in random order (fixed seed) and smaller points, per the project UMAP rule. This is the fig1c panel intended for the paper; the variations above (cells 3-9) are kept for reference.


In [ ]:
# ---- Chosen variant (the _v2 embedding above) re-plotted in RANDOM cell order ----
# Project rule: every paper UMAP is plotted with cells in random order (fixed seed) so no
# cell-type is systematically drawn on top of another. Point size reduced vs v2 (TODO 4).
# Background (non-lineage) cells kept in grey, as in the chosen v2 panel. This is the
# panel intended for the paper; cells 3-9 above are earlier variations kept for reference.
import matplotlib as mpl
from matplotlib.lines import Line2D

PANEL = "fig1c_proportions_umap_v3_randorder"
SEED = 0
S_FG, S_BG = 1.0, 0.6  # reduced from v2 (1.5 / 0.5)

xy_all = adata_prop.obsm["X_umap"]
ct_all = adata_prop.obs["celltype"].astype(str).values
lin_mask = prop_lineage.values
pt = adata_prop.obs["dpt_pseudotime"].values.astype(float)
sizes = np.where(lin_mask, S_FG, S_BG)  # lineage larger than background

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# panel 1 - cell type (lineage coloured, non-lineage grey), one random-order scatter
rgba_ct = S.palette_rgba(
    ct_all, S.paper_palette_celltypes, default=S.BG_GRAY, alpha=0.7
)
S.scatter_random_order(
    axes[0], xy_all[:, 0], xy_all[:, 1], rgba=rgba_ct, s=sizes, seed=SEED
)
handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.paper_palette_celltypes[c],
        markersize=3,
        label=c,
    )
    for c in S.MOUSE_LINEAGE_ORDER
]
axes[0].legend(
    handles=handles,
    fontsize=S.FONT_LEGEND - 0.5,
    frameon=False,
    loc="lower right",
    handletextpad=0.1,
    borderpad=0.1,
    labelspacing=0.15,
)
axes[0].set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
axes[0].set_title("Cell type", fontsize=S.FONT_LABEL, pad=2)

# panel 2 - pseudotime (lineage by magma, non-lineage grey), one random-order scatter
norm = mpl.colors.Normalize(vmin=np.nanmin(pt[lin_mask]), vmax=np.nanmax(pt[lin_mask]))
cmap = plt.get_cmap(S.PTIME_CMAP)
rgba_pt = np.tile(np.array(mpl.colors.to_rgba(S.BG_GRAY, 0.7)), (xy_all.shape[0], 1))
rgba_pt[lin_mask] = cmap(norm(pt[lin_mask]))
S.scatter_random_order(
    axes[1], xy_all[:, 0], xy_all[:, 1], rgba=rgba_pt, s=sizes, seed=SEED
)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[1], fraction=0.05, pad=0.02)
cbar.set_label("Pseudotime", fontsize=S.FONT_TICK)
cbar.ax.tick_params(labelsize=S.FONT_TICK - 1)
axes[1].set_title("Pseudotime", fontsize=S.FONT_LABEL, pad=2)

for ax in axes:
    ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

axes[0].text(
    0.50,
    0.98,
    "3'UTR proportions\n(Harmony-integrated)",
    transform=axes[0].transAxes,
    fontsize=S.FONT_ANNOT - 0.5,
    va="top",
    ha="left",
    style="italic",
)
fig.subplots_adjust(wspace=0.05, left=0.05, right=0.9, top=0.92, bottom=0.10)
S.save_and_show(fig, PANEL)

### Variant: per-mouse inner pie + title at whole-panel top-right


In [ ]:
# --- fig1c variant: per-mouse (bio-replicate) inner pie in the first UMAP + title moved to
# the whole-panel top-right (like fig1a_umap__smallpts). Reuses adata_prop / prop_lineage. ---
PANEL_PIE = "fig1c_proportions_umap_v3_randorder__pie"
SEED = 0
S_FG, S_BG = 1.0, 0.6

xy_all = adata_prop.obsm["X_umap"]
ct_all = adata_prop.obs["celltype"].astype(str).values
lin_mask = prop_lineage.values
pt = adata_prop.obs["dpt_pseudotime"].values.astype(float)
sizes = np.where(lin_mask, S_FG, S_BG)

# per bio-replicate ("mouse") lineage-cell counts for the inset pie
mouse_lin = adata_prop.obs.loc[lin_mask, "batch_number"].astype(str).values
mice = sorted(np.unique(mouse_lin), key=lambda s: int(s))
mouse_counts = np.array([(mouse_lin == m).sum() for m in mice])
MOUSE_COLORS = plt.cm.Set2(np.linspace(0, 1, len(mice)))

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

rgba_ct = S.palette_rgba(
    ct_all, S.paper_palette_celltypes, default=S.BG_GRAY, alpha=0.7
)
S.scatter_random_order(
    axes[0], xy_all[:, 0], xy_all[:, 1], rgba=rgba_ct, s=sizes, seed=SEED
)
handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.paper_palette_celltypes[c],
        markersize=3,
        label=c,
    )
    for c in S.MOUSE_LINEAGE_ORDER
]
axes[0].legend(
    handles=handles,
    fontsize=S.FONT_LEGEND - 0.5,
    frameon=False,
    loc="lower right",
    handletextpad=0.1,
    borderpad=0.1,
    labelspacing=0.15,
)
axes[0].set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
axes[0].set_title("Cell type", fontsize=S.FONT_LABEL, pad=2)

# inner pie: proportion of lineage cells contributed by each mouse (bio-replicate) - top-right
axpie = axes[0].inset_axes([0.70, 0.66, 0.30, 0.30])
axpie.pie(
    mouse_counts,
    colors=MOUSE_COLORS,
    startangle=90,
    counterclock=False,
    wedgeprops=dict(edgecolor="white", linewidth=0.3),
)
axpie.set_title(f"cells per mouse\n(n={len(mice)})", fontsize=S.FONT_ANNOT - 1.5, pad=1)

norm = mpl.colors.Normalize(vmin=np.nanmin(pt[lin_mask]), vmax=np.nanmax(pt[lin_mask]))
cmap = plt.get_cmap(S.PTIME_CMAP)
rgba_pt = np.tile(np.array(mpl.colors.to_rgba(S.BG_GRAY, 0.7)), (xy_all.shape[0], 1))
rgba_pt[lin_mask] = cmap(norm(pt[lin_mask]))
S.scatter_random_order(
    axes[1], xy_all[:, 0], xy_all[:, 1], rgba=rgba_pt, s=sizes, seed=SEED
)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[1], fraction=0.05, pad=0.02)
cbar.set_label("Pseudotime", fontsize=S.FONT_TICK)
cbar.ax.tick_params(labelsize=S.FONT_TICK - 1)
axes[1].set_title("Pseudotime", fontsize=S.FONT_LABEL, pad=2)

for ax in axes:
    ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

# title moved to the top-right of the WHOLE panel (was top of the first subpanel)
fig.text(
    0.985,
    0.985,
    "3'UTR proportions\n(Harmony-integrated)",
    ha="right",
    va="top",
    fontsize=S.FONT_ANNOT - 0.5,
    style="italic",
)
fig.subplots_adjust(wspace=0.05, left=0.05, right=0.9, top=0.88, bottom=0.10)
S.save_and_show(fig, PANEL_PIE)

### Variant: larger cell-type legend dots (only difference vs \_\_pie)


In [ ]:
# --- Same as the __pie panel, ONLY the cell-type legend marker dots are larger (markersize 3->5). ---
# --- fig1c variant: per-mouse (bio-replicate) inner pie in the first UMAP + title moved to
# the whole-panel top-right (like fig1a_umap__smallpts). Reuses adata_prop / prop_lineage. ---
PANEL_PIE = "fig1c_proportions_umap_v3_randorder__pie_biglegend"
SEED = 0
S_FG, S_BG = 1.0, 0.6

xy_all = adata_prop.obsm["X_umap"]
ct_all = adata_prop.obs["celltype"].astype(str).values
lin_mask = prop_lineage.values
pt = adata_prop.obs["dpt_pseudotime"].values.astype(float)
sizes = np.where(lin_mask, S_FG, S_BG)

# per bio-replicate ("mouse") lineage-cell counts for the inset pie
mouse_lin = adata_prop.obs.loc[lin_mask, "batch_number"].astype(str).values
mice = sorted(np.unique(mouse_lin), key=lambda s: int(s))
mouse_counts = np.array([(mouse_lin == m).sum() for m in mice])
MOUSE_COLORS = plt.cm.Set2(np.linspace(0, 1, len(mice)))

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

rgba_ct = S.palette_rgba(
    ct_all, S.paper_palette_celltypes, default=S.BG_GRAY, alpha=0.7
)
S.scatter_random_order(
    axes[0], xy_all[:, 0], xy_all[:, 1], rgba=rgba_ct, s=sizes, seed=SEED
)
handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.paper_palette_celltypes[c],
        markersize=5,
        label=c,
    )
    for c in S.MOUSE_LINEAGE_ORDER
]
axes[0].legend(
    handles=handles,
    fontsize=S.FONT_LEGEND - 0.5,
    frameon=False,
    loc="lower right",
    handletextpad=0.1,
    borderpad=0.1,
    labelspacing=0.15,
)
axes[0].set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
axes[0].set_title("Cell type", fontsize=S.FONT_LABEL, pad=2)

# inner pie: proportion of lineage cells contributed by each mouse (bio-replicate) - top-right
axpie = axes[0].inset_axes([0.70, 0.66, 0.30, 0.30])
axpie.pie(
    mouse_counts,
    colors=MOUSE_COLORS,
    startangle=90,
    counterclock=False,
    wedgeprops=dict(edgecolor="white", linewidth=0.3),
)
axpie.set_title(f"cells per mouse\n(n={len(mice)})", fontsize=S.FONT_ANNOT - 1.5, pad=1)

norm = mpl.colors.Normalize(vmin=np.nanmin(pt[lin_mask]), vmax=np.nanmax(pt[lin_mask]))
cmap = plt.get_cmap(S.PTIME_CMAP)
rgba_pt = np.tile(np.array(mpl.colors.to_rgba(S.BG_GRAY, 0.7)), (xy_all.shape[0], 1))
rgba_pt[lin_mask] = cmap(norm(pt[lin_mask]))
S.scatter_random_order(
    axes[1], xy_all[:, 0], xy_all[:, 1], rgba=rgba_pt, s=sizes, seed=SEED
)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[1], fraction=0.05, pad=0.02)
cbar.set_label("Pseudotime", fontsize=S.FONT_TICK)
cbar.ax.tick_params(labelsize=S.FONT_TICK - 1)
axes[1].set_title("Pseudotime", fontsize=S.FONT_LABEL, pad=2)

for ax in axes:
    ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

# title moved to the top-right of the WHOLE panel (was top of the first subpanel)
fig.text(
    0.985,
    0.985,
    "3'UTR proportions\n(Harmony-integrated)",
    ha="right",
    va="top",
    fontsize=S.FONT_ANNOT - 0.5,
    style="italic",
)
fig.subplots_adjust(wspace=0.05, left=0.05, right=0.9, top=0.88, bottom=0.10)
S.save_and_show(fig, PANEL_PIE)